# 🧠 ABIDE ASD Multimodal Analysis — Exploratory Notebook

**Authors:** Jarin Alam Prity (222-115-005) & Popy Rani Boidya (007)  
**Supervisor:** Md Mahfujul Hasan — Metropolitan University, Sylhet-3104  
**Clinical:** Prof. Imdadul Magfur — Sylhet MAG Osmani Medical College  

---
This notebook walks through:
1. Data loading & quality audit
2. fMRI connectivity matrix exploration
3. Track A unimodal results
4. Track B fusion results
5. Track C missing-modality robustness
6. Track D explainability (SHAP)
7. Track E fairness analysis


In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

matplotlib.rcParams['font.family'] = 'DejaVu Sans'
matplotlib.rcParams['figure.dpi'] = 120

print('✓ Imports OK')
print('asd_multimodal version:', __import__('asd_multimodal').__version__)

## 1. Data Loading & Quality Audit

In [ ]:
from asd_multimodal.data.abide_loader import (
    load_abide1_phenotypic, load_abide2_phenotypic,
    check_motion_confound
)

# ABIDE-I
df1 = load_abide1_phenotypic('../data/raw/Phenotypic_V1_0b.csv')
print('\nABIDE-I shape:', df1.shape)
print('DX_GROUP:', df1['DX_GROUP'].value_counts().to_dict())

In [ ]:
# ABIDE-II (critical: latin1 encoding, strip column names)
try:
    df2 = load_abide2_phenotypic('../data/raw/ABIDEII_Composite_Phenotypic.csv')
    print('ABIDE-II shape:', df2.shape)
    print('DX_GROUP:', df2['DX_GROUP'].value_counts().to_dict())
except FileNotFoundError:
    print('ABIDE-II file not found. Download with: bash scripts/download_abide.sh')
    df2 = None

In [ ]:
# Missing value audit
AUDIT_COLS = ['AGE_AT_SCAN','SEX','FIQ','VIQ','PIQ',
              'ADOS_TOTAL','ADI_R_SOCIAL_TOTAL_A','SRS_RAW_TOTAL']

fig, ax = plt.subplots(figsize=(10, 4))
miss_pct = {}
for col in AUDIT_COLS:
    if col in df1.columns:
        miss_pct[col] = 100 * df1[col].isna().mean()

cols_sorted = sorted(miss_pct, key=miss_pct.get, reverse=True)
vals  = [miss_pct[c] for c in cols_sorted]
colors= ['#E74C3C' if v > 60 else '#F39C12' if v > 20 else '#27AE60' for v in vals]
ax.barh(cols_sorted, vals, color=colors, alpha=0.85)
ax.set_xlabel('% Missing')
ax.set_title('Missing Value Rates — ABIDE-I\n(red>60%: MNAR/excluded; orange: moderate; green: acceptable)')
ax.axvline(20, color='orange', linestyle='--', alpha=0.5)
ax.axvline(60, color='red',    linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()
print('\nMissing rates:')
for c, v in zip(cols_sorted, vals):
    print(f'  {c:<30} {v:5.1f}%')

## 2. fMRI Connectivity Matrix

In [ ]:
import os

conn_path = '../data/processed/connectivity_matrix.npy'
meta_path = '../data/processed/connectivity_metadata.csv'

if os.path.exists(conn_path):
    X_fmri = np.load(conn_path)
    meta   = pd.read_csv(meta_path)
    y      = (meta['DX_GROUP'] == 1).astype(int).values
    sex    = meta['SEX'].values

    print(f'Connectivity matrix: {X_fmri.shape}')
    print(f'ASD: {y.sum()}  TDC: {(1-y).sum()}')
    print(f'Feature range: [{X_fmri.min():.3f}, {X_fmri.max():.3f}]')
    print(f'NaN: {np.isnan(X_fmri).sum()}')

    # Motion confound check
    motion = check_motion_confound(meta)
    print(f'\nMotion: ASD={motion["ASD_mean_FD"]:.3f}  TDC={motion["TDC_mean_FD"]:.3f}  p={motion["p_value"]:.4f}')
    if motion['significant']:
        print('  ⚠ SIGNIFICANT MOTION CONFOUND — include FD as ComBat covariate')
else:
    print('Connectivity matrix not found.')
    print('Run: python scripts/compute_connectivity.py')
    X_fmri = None; meta = None; y = None; sex = None

In [ ]:
if X_fmri is not None:
    # PCA variance analysis
    from sklearn.preprocessing import StandardScaler
    from sklearn.decomposition import PCA
    from sklearn.impute import SimpleImputer

    X_sc = StandardScaler().fit_transform(
        SimpleImputer(strategy='mean').fit_transform(X_fmri)
    )
    pca = PCA(n_components=100, random_state=42)
    pca.fit(X_sc)
    var_cumsum = np.cumsum(pca.explained_variance_ratio_) * 100

    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(range(1, 101), var_cumsum, color='#2E86AB', linewidth=2)
    ax.fill_between(range(1, 101), var_cumsum, alpha=0.15, color='#2E86AB')
    for n, c in [(30,'orange'), (50,'green'), (100,'red')]:
        ax.axvline(n, color=c, linestyle='--', alpha=0.6, linewidth=1)
        ax.text(n+1, var_cumsum[n-1]+0.5, f'{var_cumsum[n-1]:.1f}%@{n}PCs',
                fontsize=8, color=c)
    ax.set_xlabel('Number of PCs'); ax.set_ylabel('Cumulative Variance (%)')
    ax.set_title('fMRI PCA Variance Explained (CC200, 19,900 features → 100 PCs)')
    plt.tight_layout(); plt.show()
    print(f'30 PCs: {var_cumsum[29]:.1f}%  50 PCs: {var_cumsum[49]:.1f}%  100 PCs: {var_cumsum[99]:.1f}%')

## 3. Track A — Unimodal Baseline Results

In [ ]:
ta_path = '../results/track_a/track_a_results.csv'
if os.path.exists(ta_path):
    df_a = pd.read_csv(ta_path)
    df_a = df_a[df_a['status'] == 'OK']

    print('Track A Results:')
    cols = ['Modality','Model','AUC','AUC_lo','AUC_hi','BAC','Sens','Spec']
    avail = [c for c in cols if c in df_a.columns]
    print(df_a[avail].sort_values(['Modality','AUC'],ascending=[True,False]).to_string(index=False))

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    # AUC by modality
    MCOL = {'RandomForest':'#2E86AB','XGBoost':'#E84855','LightGBM':'#F4A261',
            'MLP':'#06A77D','TabTransformer':'#8B5CF6','BrainGNN':'#EC4899','GraphTransformer':'#F59E0B'}
    for i, (mod, grp) in enumerate(df_a.groupby('Modality')):
        x_pos = np.arange(len(grp))
        bars  = axes[0].bar(x_pos + i*0.1, grp['AUC'].values, 0.6,
                             label=mod, alpha=0.8)
    axes[0].axhline(0.5, color='gray', linestyle='--', alpha=0.4)
    axes[0].set_ylabel('AUC'); axes[0].set_title('AUC by Modality')
    axes[0].legend(frameon=False, fontsize=8)

    # Best per modality
    best = df_a.groupby('Modality')['AUC'].max().reset_index()
    axes[1].bar(range(len(best)), best['AUC'], color=['#2E86AB','#E05A5A','#6CB87A','#E07B54'][:len(best)], alpha=0.9)
    axes[1].set_xticks(range(len(best))); axes[1].set_xticklabels(best['Modality'], rotation=15, ha='right')
    axes[1].set_ylabel('AUC'); axes[1].set_title('Best AUC per Modality')
    axes[1].axhline(0.5, color='gray', linestyle='--', alpha=0.4)
    for i, v in enumerate(best['AUC']):
        axes[1].text(i, v+0.005, f'{v:.3f}', ha='center', fontsize=9, fontweight='bold')

    plt.suptitle('Track A: Unimodal Baselines | Metropolitan University, Sylhet', fontsize=10)
    plt.tight_layout(); plt.show()
else:
    print(f'Track A results not found at {ta_path}')
    print('Run: python experiments/track_a_unimodal.py')

## 4. Track B — Multimodal Fusion

In [ ]:
tb_path = '../results/track_b/track_b_results.csv'
if os.path.exists(tb_path):
    df_b = pd.read_csv(tb_path)
    print('Track B Results:')
    print(df_b[['Strategy','AUC','BAC','Sens','Spec']].to_string(index=False))

    # AUC comparison
    fig, ax = plt.subplots(figsize=(10, 5))
    strats = df_b['Strategy'].tolist()
    aucs   = df_b['AUC'].tolist()
    cols   = ['#1B6CA8' if 'LF' in s else '#D62728' if 'EF' in s else '#8B5CF6' for s in strats]
    y_pos  = np.arange(len(strats))
    ax.barh(y_pos, aucs, color=cols, alpha=0.87)
    ax.axvline(0.731, color='black', linestyle='--', alpha=0.5, label='BrainGNN ref')
    for i, v in enumerate(aucs):
        ax.text(v+0.001, i, f'{v:.3f}', va='center', fontsize=8.5, fontweight='bold')
    ax.set_yticks(y_pos); ax.set_yticklabels(strats)
    ax.set_xlabel('AUC'); ax.set_title('Track B: Multimodal Fusion Strategies | Metropolitan University')
    ax.legend(frameon=False)
    plt.tight_layout(); plt.show()
else:
    print(f'Track B results not found at {tb_path}')
    print('Run: python experiments/track_b_fusion.py')

## 5. Track C — Missing-Modality Robustness

In [ ]:
tc_path = '../results/track_c/track_c_summary.csv'
tc_full = '../results/track_c/track_c_full_results.csv'

if os.path.exists(tc_path):
    df_c = pd.read_csv(tc_path)
    print('Track C Summary (best strategy per scenario):')
    print(df_c[['Scenario','Description','Strategy','AUC','Delta_AUC','Pct_Retained','Degradation']].to_string(index=False))

    # Robustness curve
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    SC_COL = {'S1_All':'#27AE60','S2_Behav':'#2ECC71','S3_Pheno':'#F39C12',
              'S4_fMRI':'#E74C3C','S5_Demo':'#82E0AA','S6_Two':'#E67E22',
              'S7_Rand30':'#3498DB','S8_Extreme':'#922B21'}
    scs   = df_c['Scenario'].tolist()
    aucs  = df_c['AUC'].tolist()
    cols2 = [SC_COL.get(s, '#888') for s in scs]
    axes[0].bar(range(len(scs)), aucs, color=cols2, alpha=0.87)
    s1 = df_c[df_c['Scenario']=='S1_All']['AUC'].values
    if len(s1): axes[0].axhline(s1[0], color='green', linestyle='--', alpha=0.5, label=f'S1={s1[0]:.3f}')
    axes[0].set_xticks(range(len(scs))); axes[0].set_xticklabels(scs, rotation=35, ha='right', fontsize=8)
    axes[0].set_ylabel('AUC'); axes[0].set_title('Robustness Curve (Best Strategy)')
    axes[0].legend(frameon=False)

    # Strategy comparison (from full results)
    if os.path.exists(tc_full):
        df_cf  = pd.read_csv(tc_full)
        s4_df  = df_cf[df_cf['Scenario']=='S4_fMRI'].groupby('Strategy')['AUC'].mean().sort_values(ascending=False)
        strat_cols = {'Zero':'#E74C3C','Mean':'#E67E22','KNN':'#F1C40F',
                      'MICE':'#2ECC71','Conditional':'#1ABC9C','MAE':'#3498DB','VAE':'#9B59B6'}
        axes[1].bar(range(len(s4_df)), s4_df.values,
                     color=[strat_cols.get(s,'#888') for s in s4_df.index], alpha=0.85)
        axes[1].set_xticks(range(len(s4_df))); axes[1].set_xticklabels(s4_df.index, fontsize=9)
        axes[1].set_ylabel('AUC'); axes[1].set_title('S4 (fMRI missing): Strategy Comparison')
        for i, v in enumerate(s4_df.values):
            axes[1].text(i, v+0.001, f'{v:.3f}', ha='center', fontsize=8.5, fontweight='bold')

    plt.suptitle('Track C: Missing-Modality Robustness | Metropolitan University, Sylhet', fontsize=10)
    plt.tight_layout(); plt.show()

    # H1 evaluation
    print('\nH1 Evaluation: VAE vs Zero imputation')
    if os.path.exists(tc_full):
        for sc in ['S3_Pheno','S4_fMRI','S7_Rand30','S8_Extreme']:
            v = df_cf[(df_cf['Scenario']==sc)&(df_cf['Strategy']=='VAE')]['AUC']
            z = df_cf[(df_cf['Scenario']==sc)&(df_cf['Strategy']=='Zero')]['AUC']
            if len(v) and len(z):
                d = v.values[0] - z.values[0]
                sup = '✓ SUPPORTED' if d > 0.01 else '✗ not supported'
                print(f'  {sc}: VAE={v.values[0]:.3f}  Zero={z.values[0]:.3f}  Δ={d:+.3f}  → H1 {sup}')
else:
    print(f'Track C results not found. Run: python experiments/track_c_missing.py')

## 6. Track D — Explainability (SHAP)

In [ ]:
shap_path = '../results/track_d/shap_summary.csv'
pi_path   = '../results/track_d/pi_summary.csv'

if os.path.exists(shap_path):
    shap_df = pd.read_csv(shap_path)
    MOD_COL = {'fMRI':'#2E86AB','Pheno':'#E05A5A','Demo':'#6CB87A'}

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    # SHAP top 20
    top20 = shap_df.head(20)
    cols3 = [MOD_COL.get(m,'#888') for m in top20['Modality']]
    axes[0].barh(range(len(top20)), top20['MeanAbsSHAP'].values, color=cols3, alpha=0.85)
    axes[0].set_yticks(range(len(top20))); axes[0].set_yticklabels(top20['Feature'], fontsize=8)
    axes[0].set_xlabel('Mean |SHAP|'); axes[0].set_title('SHAP Top-20 Features')

    # Modality pie
    mod_s  = shap_df.groupby('Modality')['MeanAbsSHAP'].sum()
    colors_p = [MOD_COL.get(m,'#888') for m in mod_s.index]
    axes[1].pie(mod_s.values, labels=[f'{m}\n{100*v/mod_s.sum():.1f}%' for m,v in mod_s.items()],
                colors=colors_p, startangle=90,
                wedgeprops=dict(edgecolor='white', linewidth=1.5))
    axes[1].set_title('Modality SHAP Attribution (%)')

    plt.suptitle('Track D: Explainability | Metropolitan University, Sylhet', fontsize=10)
    plt.tight_layout(); plt.show()

    print('Modality attribution:')
    for m, v in mod_s.items():
        print(f'  {m}: {100*v/mod_s.sum():.1f}%')
    print(f'\nTop-5 features: {shap_df["Feature"].head(5).tolist()}')

    if os.path.exists(pi_path):
        pi_df = pd.read_csv(pi_path)
        from scipy.stats import spearmanr
        merged = shap_df.merge(pi_df[['Feature','PI_mean']], on='Feature', how='inner')
        rho, p = spearmanr(merged['MeanAbsSHAP'], merged['PI_mean'])
        print(f'\nSHAP vs PI agreement: Spearman ρ={rho:.3f}, p={p:.2e}')
        if rho > 0.90:
            print('  ✓ HIGH AGREEMENT — explainability results are robust')
else:
    print(f'Track D results not found. Run: python experiments/track_d_explain.py')

## 7. Track E — Fairness Analysis

In [ ]:
te_path = '../results/track_e/fairness_results.csv'

if os.path.exists(te_path):
    df_e   = pd.read_csv(te_path)
    strats = df_e['Strategy'].unique()

    print('⚠ REMINDER: All female results are EXPLORATORY (n♀_test≈24, CI≈±0.22)')
    print('\nFairness Results:')
    print(df_e[['Strategy','Group','N','AUC','Sens','Spec','FNR']].to_string(index=False))

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    x = np.arange(len(strats))
    MALE_COL = '#5A8FA8'; FEM_COL = '#D4789E'

    for col_n, title, ax in zip(
        ['AUC', 'Sens', 'FNR'],
        ['(a) AUC', '(b) Sensitivity', '(c) FNR (↓ is better)'],
        axes
    ):
        if col_n not in df_e.columns: continue
        m_vals = [df_e[(df_e['Strategy']==s)&(df_e['Group']=='Male')][col_n].values for s in strats]
        f_vals = [df_e[(df_e['Strategy']==s)&(df_e['Group']=='Female')][col_n].values for s in strats]
        m_vals = [v[0] if len(v) else np.nan for v in m_vals]
        f_vals = [v[0] if len(v) else np.nan for v in f_vals]

        ax.bar(x-0.2, m_vals, 0.38, color=MALE_COL, alpha=0.85, label='Male')
        ax.bar(x+0.2, f_vals, 0.38, color=FEM_COL,  alpha=0.85, label='Female [EXP]')
        ax.set_xticks(x); ax.set_xticklabels(strats, rotation=20, ha='right', fontsize=8)
        ax.set_title(title); ax.legend(frameon=False, fontsize=8)
        if col_n == 'Sens':
            ax.axhline(0.8, color='red', linestyle='--', alpha=0.6, label='Target ≥0.80')
        ax.set_ylim(0, 1.1)

    plt.suptitle(
        'Track E: Fairness Analysis\n'
        '⚠ Female results EXPLORATORY (n♀_test≈24)\n'
        'H5 NOT SUPPORTED — data scarcity bottleneck (n♀_ASD_train≈62)',
        fontsize=9, fontweight='bold'
    )
    plt.tight_layout(); plt.show()

    # H5 conclusion
    print('\nH5 Evaluation: Sex-aware modelling reduces female FNR')
    e1_fnr = df_e[(df_e['Strategy'].str.contains('E1'))&(df_e['Group']=='Female')]['FNR'].values
    if len(e1_fnr):
        print(f'  E1 (baseline) Female FNR = {e1_fnr[0]:.3f}')
        for strat in strats:
            fnr = df_e[(df_e['Strategy']==strat)&(df_e['Group']=='Female')]['FNR'].values
            if len(fnr):
                change = fnr[0] - e1_fnr[0]
                flag = '✓ improved' if change < -0.05 else '✗ not improved'
                print(f'  {strat}: FNR={fnr[0]:.3f}  Δ={change:+.3f}  {flag}')
    print('\nCONCLUSION: H5 NOT SUPPORTED.')
    print('Root cause: n♀_ASD_train≈62 (data scarcity, not algorithm failure).')
    print('Fix: Collect n♀_ASD ≥ 500 across sites.')
else:
    print(f'Track E results not found. Run: python experiments/track_e_fairness.py')

---
## Summary

| Track | Key Finding | Status |
|-------|-------------|--------|
| A | fMRI best: BrainGNN AUC=0.731 | ✓ |
| B | LF Stacked AUC=0.763 (+0.032, p<0.001) | ✓ |
| C | VAE +0.081 for random miss; 0.000 for structural | H1 partial |
| D | VIQ #1 feature; method agreement ρ=0.94 | ✓ |
| E | Female FNR=0.500 (all E1-E4 fail) | H5 rejected |

**Authors:** Jarin Alam Prity & Popy Rani Boidya  
**Metropolitan University, Dept. of CSE, Sylhet-3104, Bangladesh**  
**Supervisor:** Md Mahfujul Hasan · **Clinical:** Prof. Imdadul Magfur
